In [4]:
# Automatically restart the Jupyter kernel when dependencies change.
%load_ext autoreload
%autoreload 2
from ai_learning.dataloader import create_dataloader
import torch


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Use the `create_dataloader` function to get some (input, target) training pairs from `the-verdict.txt`.

Also a basic test to make sure our package dependenices and external dependencies are wired up correctly.

Contains my verbose and potentially wrong comments.. There's a VSCode extenson that live-renders
latex in comments, if you're interested it's called "Comment Formula."

In [ ]:
with open("../data/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Size of the dictionary, depends on the tokenizer. Torch calls it num_embeddings.
# $|V| = 50,257$ where $V = \{ \text{vocabulary tokens} \}$
vocab_size = 50_257

# Dimensionality of embedding vectors, Torch calls it embedding_dim.
# ie $\mathbf e_i \in \mathbb{R}^{d}$ and here output_dim = $d=256$.
output_dim = 256

# A table to lookup the d=256 dimension embedding vector for each of the V=50257 tokens.
# We can view a token as a one-hot vector $\mathbf x_i \in \mathbb{R}^{|V|}$. It's often denoted by the indicator vector to
# be explicit that it's all 0's except for exactly one 1: $\mathbf x_i = 1_V(i)$ or $\mathbf x_i = 1_i$.
# The embedding matrix $E \in \mathbb{R}^{|V| \times d}$ just contains as rows the embeddings for each token.
# I.e. $\mathbf e_i = \mathbf x_iE$ if we assume $\mathbf x_i$ is a row-vector.
# Here $E \in \mathbb{R}^{50257 \times 256}$.
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)  #
# Note that from the a theoretical perspective, we think of the one hot vector $\mathbf x_i$ as the input to
# the network (a sample). We let someone else (the tokenizer) figure out how to get a one-hot
# vector. But we're going to learn the embedding matrix, so it's part of the model - we don't take
# embedding vectors as inputs, we take one-hot vectors.

# max_length = $T \in Z^+$. Max number of tokens per sample, aka sequence length.
max_length = 4
dataloader = create_dataloader(
    raw_text, batch_size=8, shuffle=False, max_length=max_length, stride=max_length
)
data_iter = iter(dataloader)
# For absolute positional embedding, context_length = $T$.
context_length = max_length
# pos_embedding_layer = $P \in \mathbb{R}^{T \times d}$. We have one (learned) embedding for every possible position in
# the sample.
# Here $P \in \mathbb{R}^{4 \times 256}$.
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)


# Compute pos_embeddings = $$\mathbf p_t = \begin{bmatrix} 1_1 \\ 1_2 \\ \vdots \\ 1_T \end{bmatrix}P$$.


pos_embeddings = pos_embedding_layer(torch.arange(context_length))

# inputs is a batch matrix where each row is sample of token_ids.
inputs, targets = next(data_iter)
print(f"inputs shape = {inputs.shape}")
print(f"inputs = \n{inputs}")
print(f"targets = \n{targets}")


# Compute input_embeddings = $H^{(0)} = \tilde{X}E + P$ where $\tilde X \in \mathbb{R}^{T \times |V|}$ has as rows the one-hot token vectors.
# $E \in \mathbb{R}^{|V| \times d}$ so $\tilde X E \in \mathbb{R}^{T \times d} \implies H^{(0)} \in \mathbb{R}^{T \times d}$
token_embeddings = token_embedding_layer(inputs)
# input_embeddings = batch of $B$ inputs, where each input $H^{(0)}$ where $\mathbf h_t^{(0)} = \mathbf x_tE + \mathbf p_t$
# input_embeddings dimension = B x T x d = 8 x 4 x 256.
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

inputs shape = torch.Size([8, 4])
inputs = 
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets = 
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
torch.Size([8, 4, 256])
